# MLOps Overview — Exercises

Companion to the note [MLOps Overview](MLOps%20Overview.md).

Practise keeping models healthy in production: kinds of drift and how to detect them (KS test, PSI), A/B testing statistics, serving capacity, data validation, reproducibility (data fingerprints, seeds) and behavioural tests. Leans conceptual, with small NumPy / pandas / SciPy implementations of the monitoring and testing tools.

**17 exercises** · 🧠 Concept ×5 · ✍️ By hand ×5 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import hashlib
import pandas as pd
from math import ceil
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
rng = np.random.default_rng(0)

## Part A · Concepts

### Exercise 1 · Data drift, concept drift, label shift
*🧠 Concept · ★☆☆*

Classify each scenario as **data (covariate) drift** ($P(X)$ changes, $P(y\mid X)$ fixed), **concept drift** ($P(y\mid X)$ changes) or
**label shift** ($P(y)$ changes, $P(X\mid y)$ fixed), and say what you would monitor:

(a) A fraud model: fraudsters change tactics after the model blocks their old pattern.
(b) A medical image classifier deployed at a hospital with a new scanner brand.
(c) A flu classifier during flu season: many more positive patients, symptoms per class unchanged.
(d) A demand forecaster after a competitor opens next door: same customers, different buying behaviour.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: did the inputs change, the relationship between inputs and outcome, or only the class proportions?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Concept drift** (adversarial): same-looking transactions now have different labels. Monitor performance on delayed labels (chargebacks), and the prediction distribution.
(b) **Data drift**: pixel statistics change, the disease→appearance relation does not (ideally). Monitor input statistics/embedding drift (KS, PSI, MMD).
(c) **Label shift**: prior $P(y)$ rises. Monitor the predicted-positive rate; recalibrate the threshold/prior.
(d) **Concept drift**: the same features map to different demand. Monitor forecast error once actuals arrive.
Key point: input-drift monitors cannot see pure concept drift (see the drift-simulation exercise in Part C); you need labels, possibly delayed.

</details>

### Exercise 2 · Batch or online serving?
*🧠 Concept · ★☆☆*

Choose batch (precompute predictions periodically) or online (compute on request) serving and justify:
(a) nightly churn scores for a marketing email campaign, (b) credit-card fraud check at payment time,
(c) "recommended for you" on a homepage with 10M users, (d) search-query autocomplete.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Do the inputs exist before the request? How fresh must the prediction be? What is the latency budget?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Batch**: the decision is made once a day; cheapest, no latency constraint.
(b) **Online**: the transaction features only exist at request time, and the latency budget is tens of milliseconds.
(c) Often **hybrid**: batch-precompute candidate lists per user overnight, then rerank online with session context.
(d) **Online**, with an extremely tight budget (< 50 ms per keystroke), so a small model plus caching of popular prefixes.
Batch pitfalls: stale predictions and wasted compute for users who never show up. Online pitfalls: latency, availability, training–serving skew.

</details>

### Exercise 3 · Shadow, canary, A/B
*🧠 Concept · ★★☆*

Explain shadow deployment, canary release and A/B testing. For each: do users see the new model's output? What question does it answer?
In what order would you use them when replacing a production ranking model?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

One of them sends traffic to the new model but throws away its answers.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Shadow**: the new model receives a copy of live traffic, its outputs are logged but **not shown**. Answers "does it run correctly
at production load (latency, errors), and how do its predictions differ from the old model's?" with zero user risk.
**Canary**: a small slice (1–5 %) of users get the new model; automatic rollback if error/latency/business metrics degrade. Answers "is it safe?"
**A/B test**: a randomised split (e.g. 50/50) with a pre-registered metric and sample size. Answers "is it *better*, with statistical significance?"
Typical order: offline evaluation → shadow → canary → A/B → full rollout (keeping the old model ready for rollback).

</details>

### Exercise 4 · Training–serving skew
*🧠 Concept · ★★☆*

A model reaches 0.92 AUC offline but 0.70 in production, with no drift in the raw data. Give three concrete causes of
**training–serving skew** and explain how a feature store (e.g. Feast) helps with two of them.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compare how a feature is computed in the training notebook vs the serving code path, and *when* its value is known.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Causes: (1) **different code paths**: features computed in SQL/pandas for training, re-implemented in Java/Python for serving, with subtle
differences (time zones, missing-value handling, unit conversions); (2) **leakage / point-in-time errors**: training used feature values
that were only known *after* the prediction time (e.g. "number of purchases this month" computed at month end); (3) **stale or missing
features online** (a lookup times out and a default is used), (4) different preprocessing versions (a scaler re-fit).
A feature store defines each feature **once** and serves the same values to training (offline store) and inference (online store), and
provides **point-in-time-correct joins** for building training sets, addressing (1) and (2). See [[Common Pitfalls]].

</details>

### Exercise 5 · Reproducibility and governance
*🧠 Concept · ★☆☆*

Six months after deployment, a regulator asks you to reproduce exactly how model v3 was trained and why it denied an application.
List what you must have recorded at training time (think of the note's Data, Experiments, Packaging and Governance rows) and what a
**model card** contains.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Code, data, configuration, environment, randomness, and the evaluation.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Record: the **code version** (git commit), the **data version** (DVC/lakeFS hash of the exact training snapshot), **config and
hyperparameters** (Hydra), **random seeds**, the **environment** (Docker image / lock file with library versions, hardware), the trained
**artifact** in a model registry with its metrics, and the **lineage** linking all of these (MLflow/W&B run). For individual decisions, log
model version, inputs and outputs per prediction (and an explanation, see [[Explainability and Fairness]]).
A **model card** documents intended use and out-of-scope uses, training data, evaluation metrics overall and **per subgroup**, known
limitations and ethical considerations.

</details>

## Part B · By hand

### Exercise 6 · Population Stability Index
*✍️ By hand · ★☆☆*

A feature is binned into three bins. Training (expected) proportions: $e = (0.25, 0.25, 0.5)$; this week (actual): $a = (0.1, 0.3, 0.6)$.
Compute $\text{PSI} = \sum_i (a_i - e_i)\ln\frac{a_i}{e_i}$. A common rule of thumb: $<0.1$ stable, $0.1$–$0.25$ moderate, $>0.25$ major shift.

In [ ]:
psi_hand = None

_e, _a = np.array([.25, .25, .5]), np.array([.1, .3, .6])
check('PSI', psi_hand, np.sum((_a - _e) * np.log(_a / _e)))

<details>
<summary><b>💡 Hint</b></summary>

Terms: $(-0.15)\ln 0.4$, $(0.05)\ln 1.2$, $(0.1)\ln 1.2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(-0.15)(-0.9163) + 0.05(0.1823) + 0.1(0.1823) = 0.1374 + 0.0091 + 0.0182 \approx 0.165$: a **moderate** shift, worth investigating.
PSI is a symmetrised KL divergence: $\text{PSI} = \mathrm{KL}(a\|e) + \mathrm{KL}(e\|a)$. Empty bins make it infinite, so add a small $\epsilon$.

```python
psi_hand = 0.15 * np.log(2.5) + 0.15 * np.log(1.2)
```

</details>

### Exercise 7 · Kolmogorov–Smirnov statistic by hand
*✍️ By hand · ★☆☆*

Reference sample $\{1, 2, 3, 4\}$, production sample $\{3, 4, 5, 6\}$. Sketch both empirical CDFs and compute the two-sample KS statistic
$D = \sup_x |F_{ref}(x) - F_{prod}(x)|$.

In [ ]:
ks_D = None

check('KS statistic', ks_D, stats.ks_2samp([1, 2, 3, 4], [3, 4, 5, 6]).statistic)

<details>
<summary><b>💡 Hint</b></summary>

Evaluate both ECDFs just after each of the points 1, …, 6.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| $x$ | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $F_{ref}$ | .25 | .5 | .75 | 1 | 1 | 1 |
| $F_{prod}$ | 0 | 0 | .25 | .5 | .75 | 1 |
| diff | .25 | .5 | .5 | .5 | .25 | 0 |

$D = 0.5$. With only 4 points per sample this is not significant ($p \approx 0.77$): KS needs reasonably large samples, and with millions
of rows it flags even tiny, harmless shifts, so pair the p-value with an effect size (like $D$ or PSI).

```python
ks_D = 0.5
```

</details>

### Exercise 8 · Is the A/B result significant?
*✍️ By hand · ★★☆*

Control: 500 conversions out of 5000 users. Treatment (new model): 560 out of 5000. Compute the two-proportion z statistic with the
pooled proportion, $z = \frac{\hat p_T - \hat p_C}{\sqrt{\hat p(1-\hat p)(1/n_C + 1/n_T)}}$, and the two-sided p-value. Ship it?

In [ ]:
z_ab = None
p_ab = None

_chi2 = stats.chi2_contingency([[560, 4440], [500, 4500]], correction=False)
check('z', z_ab, np.sqrt(_chi2[0]), tol=1e-3)
check('p-value', p_ab, _chi2[1], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\hat p = 1060/10000 = 0.106$; the standard error is $\sqrt{0.106\cdot0.894\cdot 0.0004}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\hat p_C = 0.10$, $\hat p_T = 0.112$, SE $= \sqrt{0.094764\cdot0.0004} \approx 0.006157$, so $z \approx 1.949$ and $p = 2(1 - \Phi(1.949)) \approx 0.051$.
Not significant at $\alpha = 0.05$. Do **not** keep the test running until it becomes significant ("peeking" inflates false positives);
decide the sample size in advance (next exercise) or use a sequential testing method.

```python
_p = 1060 / 10000
z_ab = 0.012 / np.sqrt(_p * (1 - _p) * (2 / 5000))
p_ab = 2 * (1 - stats.norm.cdf(z_ab))
```

</details>

### Exercise 9 · Sample size for an A/B test
*✍️ By hand · ★★☆*

Baseline conversion is 10 %. You want to detect an absolute lift of 1 percentage point (to 11 %) with a two-sided $\alpha = 0.05$ and power
$0.8$. Use $n = \frac{(z_{1-\alpha/2} + z_{1-\beta})^2\,(p_1(1-p_1) + p_2(1-p_2))}{(p_2 - p_1)^2}$ per arm.

In [ ]:
n_per_arm = None   # round up to an integer

_z = stats.norm.ppf(0.975) + stats.norm.ppf(0.8)
check('n per arm', n_per_arm, ceil(_z**2 * (0.1*0.9 + 0.11*0.89) / 0.01**2), tol=1)

<details>
<summary><b>💡 Hint</b></summary>

$z_{0.975} \approx 1.960$, $z_{0.8} \approx 0.842$, so $(z_{0.975}+z_{0.8})^2 \approx 7.849$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$7.849\cdot(0.09 + 0.0979)/0.0001 = 7.849\cdot1879 \approx 14{,}748$ users **per arm**. Halving the detectable effect quadruples $n$:
small lifts need very large experiments, which is why many A/B tests are underpowered. The previous exercise had only 5000 per arm,
so a 1.2-point lift being borderline is exactly what to expect.

```python
_z = stats.norm.ppf(0.975) + stats.norm.ppf(0.8)
n_per_arm = ceil(_z**2 * (0.1 * 0.9 + 0.11 * 0.89) / 0.01**2)
```

</details>

### Exercise 10 · Capacity planning with Little's law
*✍️ By hand · ★☆☆*

An online model receives $\lambda = 400$ requests/s; each request occupies a worker for $W = 25$ ms. Little's law says the average number of
requests in service is $L = \lambda W$. How many single-request workers do you need on average, and how many if you want each at most 70 %
utilised (headroom for spikes and tail latency)?

In [ ]:
workers_avg = None
workers_70 = None

check('average busy workers', workers_avg, 400 * 0.025)
check('workers at 70% utilisation', workers_70, ceil(400 * 0.025 / 0.7))

<details>
<summary><b>💡 Hint</b></summary>

$L = 400 \cdot 0.025$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$L = 400\cdot0.025 = 10$ busy workers on average; at 70 % utilisation you need $\lceil 10/0.7\rceil = 15$. Running close to 100 % makes queues
(and p99 latency) explode, the reason latency budgets are stated as percentiles. Dynamic batching on a GPU changes the picture: one worker
serves many requests per forward pass, trading a little latency for throughput.

```python
workers_avg = 10
workers_70 = 15
```

</details>

## Part C · Code

### Exercise 11 · Two-sample KS test from scratch
*💻 Code · ★★☆*

Implement `ks_statistic(a, b)` computing $D = \max_x|F_a(x) - F_b(x)|$ by evaluating both ECDFs at all points of the pooled sample
(`np.searchsorted(..., side='right')`). Compare with `scipy.stats.ks_2samp` on a reference sample and a slightly shifted production sample.

In [ ]:
def ks_statistic(a, b):
    # TODO
    return None

ref_sample = rng.normal(0, 1, 2000)
prod_sample = rng.normal(0.15, 1, 2000)
D_mine = ks_statistic(ref_sample, prod_sample)

check('KS statistic', D_mine, stats.ks_2samp(ref_sample, prod_sample).statistic)
check('by-hand example', ks_statistic([1,2,3,4], [3,4,5,6]), 0.5)

<details>
<summary><b>💡 Hint</b></summary>

`F_a = np.searchsorted(np.sort(a), x, side='right') / len(a)` for all pooled points `x`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The supremum is attained at one of the sample points, so evaluating at the pooled sample is exact. For a shift of 0.15 standard deviations
with 2000 points per side, $D\approx 0.06$–$0.08$ and the p-value is tiny: statistically clear, practically small. In monitoring, run such a test
per feature on a schedule, and correct for testing many features (e.g. Bonferroni) to avoid constant false alarms.

```python
def ks_statistic(a, b):
    a, b = np.sort(np.asarray(a, float)), np.sort(np.asarray(b, float))
    x = np.concatenate([a, b])
    Fa = np.searchsorted(a, x, side='right') / len(a)
    Fb = np.searchsorted(b, x, side='right') / len(b)
    return np.max(np.abs(Fa - Fb))

ref_sample = rng.normal(0, 1, 2000)
prod_sample = rng.normal(0.15, 1, 2000)
D_mine = ks_statistic(ref_sample, prod_sample)
```

</details>

### Exercise 12 · A PSI drift monitor
*💻 Code · ★★☆*

Implement `psi(ref, cur, n_bins=10, eps=1e-6)` that bins both samples using **quantile** bin edges of the reference sample (so each reference
bin holds ~10 %), and returns the PSI. Apply it to every column of the two DataFrames below and store the names of the columns with
PSI $> 0.25$ in `drifted` (a sorted list).

In [ ]:
n = 5000
df_ref = pd.DataFrame({'age': rng.normal(40, 10, n), 'income': rng.lognormal(10, 0.5, n), 'clicks': rng.poisson(3, n)})
df_cur = pd.DataFrame({'age': rng.normal(40, 10, n), 'income': rng.lognormal(10.4, 0.5, n), 'clicks': rng.poisson(3, n)})

def psi(ref, cur, n_bins=10, eps=1e-6):
    # TODO
    return None

drifted = None

check('by-hand example (3 bins)', psi(np.r_[np.zeros(25), np.ones(25), 2 * np.ones(50)], np.r_[np.zeros(10), np.ones(30), 2 * np.ones(60)], n_bins=3), 0.15 * np.log(2.5) + 0.15 * np.log(1.2), tol=1e-3)
check('drifted columns', drifted, ['income'])

<details>
<summary><b>💡 Hint 1</b></summary>

Edges: `np.unique(np.quantile(ref, np.linspace(0, 1, n_bins + 1)))`, then set the first/last edge to $-\infty$/$+\infty$ so that new extremes are counted.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`np.unique` matters for discrete features (clicks): repeated quantiles would create empty bins.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Only `income` drifts (its log-mean moved by 0.4, i.e. ~50 % higher incomes): PSI is well above 0.25, while `age` and `clicks` stay near 0.
Quantile bins make PSI scale-free and well-defined for skewed features. For categorical features use the categories as bins.

```python
n = 5000
df_ref = pd.DataFrame({'age': rng.normal(40, 10, n), 'income': rng.lognormal(10, 0.5, n), 'clicks': rng.poisson(3, n)})
df_cur = pd.DataFrame({'age': rng.normal(40, 10, n), 'income': rng.lognormal(10.4, 0.5, n), 'clicks': rng.poisson(3, n)})

def psi(ref, cur, n_bins=10, eps=1e-6):
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, n_bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(ref, edges)[0] / len(ref) + eps
    a = np.histogram(cur, edges)[0] / len(cur) + eps
    return np.sum((a - e) * np.log(a / e))

scores = {c: psi(df_ref[c].values, df_cur[c].values) for c in df_ref}
print({k: round(v, 3) for k, v in scores.items()})
drifted = sorted(c for c, v in scores.items() if v > 0.25)
```

</details>

### Exercise 13 · Data validation checks
*💻 Code · ★☆☆*

Write a tiny Great-Expectations-style validator `validate(df, schema)` returning a list of human-readable failures. The schema supports, per
column: `'range': (lo, hi)`, `'max_null_frac': f`, `'allowed': set(...)`. Run it on a clean and a corrupted batch. How many checks fail on the
corrupted batch?

In [ ]:
schema = {'age': {'range': (0, 120), 'max_null_frac': 0.0},
          'income': {'range': (0, np.inf), 'max_null_frac': 0.1},
          'country': {'allowed': {'DK', 'SE', 'NO'}}}
good_df = pd.DataFrame({'age': [25, 40, 61, 33, 50], 'income': [3e5, 4.2e5, 4.4e5, 5e5, 3.8e5],
                        'country': ['DK', 'SE', 'NO', 'DK', 'DK']})
bad_df = pd.DataFrame({'age': [25, -5, 61, 150, 50], 'income': [3e5, np.nan, np.nan, 5e5, 3.8e5],
                       'country': ['DK', 'SE', 'XX', 'DK', 'DK']})

def validate(df, schema):
    # TODO: return a list of strings, one per failed check
    return None

fails_good = validate(good_df, schema)
fails_bad = validate(bad_df, schema)
print(fails_bad)

check('clean batch passes', None if fails_good is None else len(fails_good), 0)
check('corrupted batch: number of failed checks', None if fails_bad is None else len(fails_bad), 3)

<details>
<summary><b>💡 Hint</b></summary>

For ranges, ignore NaNs (`s.dropna()`), since missingness is checked separately by `max_null_frac`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Three checks fail: `age` out of range (-5 and 150), `income` null fraction 0.4 > 0.1, and `country` contains `'XX'`. In a pipeline such a
validator runs **before** training and before batch scoring, and a failure stops the pipeline (or quarantines the batch) instead of silently
producing a bad model: "unit tests for data".

```python
schema = {'age': {'range': (0, 120), 'max_null_frac': 0.0},
          'income': {'range': (0, np.inf), 'max_null_frac': 0.1},
          'country': {'allowed': {'DK', 'SE', 'NO'}}}
good_df = pd.DataFrame({'age': [25, 40, 61, 33, 50], 'income': [3e5, 4.2e5, 4.4e5, 5e5, 3.8e5],
                        'country': ['DK', 'SE', 'NO', 'DK', 'DK']})
bad_df = pd.DataFrame({'age': [25, -5, 61, 150, 50], 'income': [3e5, np.nan, np.nan, 5e5, 3.8e5],
                       'country': ['DK', 'SE', 'XX', 'DK', 'DK']})

def validate(df, schema):
    fails = []
    for col, rules in schema.items():
        s = df[col]
        if 'range' in rules:
            lo, hi = rules['range']
            bad = s.dropna()[(s.dropna() < lo) | (s.dropna() > hi)]
            if len(bad):
                fails.append(f"{col}: {len(bad)} values outside [{lo}, {hi}]")
        if 'max_null_frac' in rules and s.isna().mean() > rules['max_null_frac']:
            fails.append(f"{col}: null fraction {s.isna().mean():.2f} > {rules['max_null_frac']}")
        if 'allowed' in rules:
            extra = set(s.dropna()) - rules['allowed']
            if extra:
                fails.append(f"{col}: unexpected values {sorted(extra)}")
    return fails

fails_good = validate(good_df, schema)
fails_bad = validate(bad_df, schema)
print(fails_bad)
```

</details>

### Exercise 14 · Data fingerprints and seeds
*💻 Code · ★☆☆*

1. Write `fingerprint(df)` returning the SHA-256 hex digest of the DataFrame's CSV serialisation (this is what DVC-style data versioning
does with file contents). Show that changing a single cell changes the hash.
2. Train `RandomForestClassifier` twice with the same `random_state` and once with a different one on the data below; store whether the
predicted probabilities are identical in `same_seed_identical` and `diff_seed_identical`.

In [ ]:
def fingerprint(df):
    # TODO
    return None

Xr = rng.normal(size=(300, 5)); yr = (Xr[:, 0] + 0.5 * rng.normal(size=300) > 0).astype(int)
df_v1 = pd.DataFrame(Xr, columns=list('abcde'))
df_v2 = df_v1.copy(); df_v2.iloc[17, 2] += 1e-9
same_seed_identical = None
diff_seed_identical = None

_h1 = fingerprint(df_v1)
check('hash is 64 hex characters', None if _h1 is None else len(_h1) == 64 and _h1 == hashlib.sha256(df_v1.to_csv(index=False).encode()).hexdigest(), True)
check('tiny edit changes the hash', None if _h1 is None else _h1 != fingerprint(df_v2), True)
check('same seed → identical', same_seed_identical, True)
check('different seed → different', diff_seed_identical, False)

<details>
<summary><b>💡 Hint</b></summary>

`hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()`. Compare probabilities with `np.array_equal`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

A change of $10^{-9}$ in one cell yields a completely different hash, so the hash identifies the exact data snapshot a model was trained on.
(Caveat: CSV float formatting means a change below the printed precision would be invisible; hashing the raw file bytes or Parquet avoids this.)
With fixed seeds, training is bit-for-bit reproducible on the same machine and library versions; a different seed changes the forest. On GPUs,
some operations are non-deterministic even with fixed seeds unless deterministic algorithms are forced.

```python
def fingerprint(df):
    return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()

Xr = rng.normal(size=(300, 5)); yr = (Xr[:, 0] + 0.5 * rng.normal(size=300) > 0).astype(int)
df_v1 = pd.DataFrame(Xr, columns=list('abcde'))
df_v2 = df_v1.copy(); df_v2.iloc[17, 2] += 1e-9
P = [RandomForestClassifier(n_estimators=30, random_state=s).fit(Xr, yr).predict_proba(Xr) for s in (1, 1, 2)]
same_seed_identical = bool(np.array_equal(P[0], P[1]))
diff_seed_identical = bool(np.array_equal(P[0], P[2]))
```

</details>

### Exercise 15 · Concept drift is invisible to input monitors
*💻 Code · ★★★*

Simulate 10 weekly windows of 1000 points each with $x\sim\mathcal N(0, I_2)$ (the **inputs never change**) and labels
$y = \mathbb 1[w_t^\top x > 0]$ where $w_t$ rotates from angle $0$ to $\pi/2$ over the windows (concept drift).
Train a logistic regression on window 0 and store its accuracy on each window in `acc_by_week` (length 10). Also run your KS statistic
(or `stats.ks_2samp`) on feature 0 between window 0 and each later window and store the maximum statistic in `max_ks`.
What would an input-drift dashboard show, and what would catch the problem?

In [ ]:
weeks = []
r3 = np.random.default_rng(3)
for t, ang in enumerate(np.linspace(0, np.pi / 2, 10)):
    Xw = r3.normal(size=(1000, 2))
    yw = (Xw @ np.array([np.cos(ang), np.sin(ang)]) > 0).astype(int)
    weeks.append((Xw, yw))

acc_by_week = None
max_ks = None

check('accuracy decays by > 0.3', None if acc_by_week is None else acc_by_week[0] - acc_by_week[-1] > 0.3, True)
check('inputs show no drift (max KS < 0.1)', None if max_ks is None else max_ks < 0.1, True)

<details>
<summary><b>💡 Hint</b></summary>

`clf = LogisticRegression().fit(*weeks[0])`; then `clf.score(Xw, yw)` per week. The accuracy at angle $\phi$ is about $1 - \phi/\pi$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Accuracy falls from ≈1.0 to ≈0.5 (for two half-planes at angle $\phi$ the agreement is $1-\phi/\pi$), while the KS statistics stay at noise
level (the largest of the 9 comparisons is ≈0.07, consistent with sampling noise for $n = 1000$, since the inputs are identically distributed): an input-drift dashboard would be green throughout. Only **performance monitoring on labels** (often delayed) catches concept
drift; proxies include the prediction distribution and user feedback signals. Remedies: scheduled or triggered retraining on recent data, and
weighting recent data more.

```python
weeks = []
r3 = np.random.default_rng(3)
for t, ang in enumerate(np.linspace(0, np.pi / 2, 10)):
    Xw = r3.normal(size=(1000, 2))
    yw = (Xw @ np.array([np.cos(ang), np.sin(ang)]) > 0).astype(int)
    weeks.append((Xw, yw))

clf = LogisticRegression().fit(*weeks[0])
acc_by_week = [clf.score(Xw, yw) for Xw, yw in weeks]
max_ks = max(stats.ks_2samp(weeks[0][0][:, 0], Xw[:, 0]).statistic for Xw, _ in weeks[1:])
print(np.round(acc_by_week, 3), round(max_ks, 3))
```

</details>

### Exercise 16 · An A/B test function
*💻 Code · ★★☆*

Implement `ab_test(conv_c, n_c, conv_t, n_t)` returning `(lift, z, p_value, ci95)` where `lift` $=\hat p_T - \hat p_C$, `z` and `p_value` use the
pooled two-proportion z-test, and `ci95` is the **unpooled** Wald interval $\text{lift}\pm1.96\sqrt{\hat p_C(1-\hat p_C)/n_C + \hat p_T(1-\hat p_T)/n_T}$.
Reproduce the Part B example, then simulate 2000 A/A tests (both arms 10 %, 5000 users each) and store the false-positive rate at
$\alpha = 0.05$ in `aa_fpr`.

In [ ]:
def ab_test(conv_c, n_c, conv_t, n_t):
    # TODO
    return None

ab = ab_test(500, 5000, 560, 5000)
aa_fpr = None

_chi2 = stats.chi2_contingency([[560, 4440], [500, 4500]], correction=False)
check('p-value (vs chi-squared test)', None if ab is None else ab[2], _chi2[1], tol=1e-4)
check('CI contains 0', None if ab is None else ab[3][0] < 0 < ab[3][1], True)
check('A/A false-positive rate ≈ 5%', aa_fpr, 0.05, tol=0.015)

<details>
<summary><b>💡 Hint</b></summary>

The z-test for two proportions is equivalent to the $\chi^2$ test on the 2×2 table: $z^2 = \chi^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$z\approx1.949$, $p\approx0.051$; the 95 % CI for the lift is about $[-0.0001, 0.0241]$, just including 0, consistent with the test.
A/A tests reject about 5 % of the time, exactly $\alpha$: running A/A tests is a standard sanity check of an experimentation platform
(randomisation bugs show up as inflated rates).

```python
def ab_test(conv_c, n_c, conv_t, n_t):
    pc, pt = conv_c / n_c, conv_t / n_t
    p = (conv_c + conv_t) / (n_c + n_t)
    z = (pt - pc) / np.sqrt(p * (1 - p) * (1 / n_c + 1 / n_t))
    pval = 2 * stats.norm.sf(abs(z))
    se = np.sqrt(pc * (1 - pc) / n_c + pt * (1 - pt) / n_t)
    return pt - pc, z, pval, (pt - pc - 1.96 * se, pt - pc + 1.96 * se)

ab = ab_test(500, 5000, 560, 5000)
r4 = np.random.default_rng(4)
aa_fpr = np.mean([ab_test(r4.binomial(5000, 0.1), 5000, r4.binomial(5000, 0.1), 5000)[2] < 0.05 for _ in range(2000)])
print(ab, aa_fpr)
```

</details>

### Exercise 17 · Behavioural tests for a model
*💻 Code · ★★☆*

Beyond accuracy, the note lists **model behavioural tests** (Ribeiro et al., "CheckList"). For the credit model below implement:
- `invariance_test(model, X, col, delta)`: max absolute change in predicted probability when column `col` is shifted by `delta`
  (should be ≈0 for an irrelevant feature);
- `directional_test(model, X, col, delta)`: fraction of rows whose probability does **not decrease** when `col` increases
  (should be 1 for income).

Store `inv_noise` (column 2, delta 0.1) and `dir_income` (column 0, delta 1.0).

In [ ]:
Xcr = rng.normal(size=(2000, 3))            # columns: income, debt, irrelevant id-like noise
ycr = (1.5 * Xcr[:, 0] - 1.0 * Xcr[:, 1] + 0.5 * rng.normal(size=2000) > 0).astype(int)
credit_model = LogisticRegression().fit(Xcr, ycr)

def invariance_test(model, X, col, delta):
    # TODO
    return None

def directional_test(model, X, col, delta):
    # TODO
    return None

inv_noise = invariance_test(credit_model, Xcr, 2, 0.1)
dir_income = directional_test(credit_model, Xcr, 0, 1.0)

check('invariance to irrelevant feature (< 0.02)', None if inv_noise is None else inv_noise < 0.02, True)
check('income is monotone', dir_income, 1.0)

<details>
<summary><b>💡 Hint</b></summary>

Copy `X`, add `delta` to one column, and compare `predict_proba(...)[:, 1]` before and after.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The irrelevant feature changes probabilities by well under 0.02 (its learned weight is close to 0), and raising income never lowers the
approval probability (a logistic regression is monotone in each feature, with positive weight on income). For flexible models (boosted trees,
neural nets) such tests can fail and catch real bugs or spurious correlations; they run in CI like unit tests, alongside slice-based
evaluation on critical subgroups ([[Explainability and Fairness]]).

```python
Xcr = rng.normal(size=(2000, 3))
ycr = (1.5 * Xcr[:, 0] - 1.0 * Xcr[:, 1] + 0.5 * rng.normal(size=2000) > 0).astype(int)
credit_model = LogisticRegression().fit(Xcr, ycr)

def _shift(X, col, delta):
    X2 = X.copy(); X2[:, col] += delta
    return X2

def invariance_test(model, X, col, delta):
    return np.max(np.abs(model.predict_proba(_shift(X, col, delta))[:, 1] - model.predict_proba(X)[:, 1]))

def directional_test(model, X, col, delta):
    return np.mean(model.predict_proba(_shift(X, col, delta))[:, 1] >= model.predict_proba(X)[:, 1])

inv_noise = invariance_test(credit_model, Xcr, 2, 0.1)
dir_income = directional_test(credit_model, Xcr, 0, 1.0)
print(inv_noise, dir_income, credit_model.coef_)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [MLOps Overview](MLOps%20Overview.md).*